# LWR_grafo. Anexo: la red de artistas detrás de los perfiles

Módulo aparte del flujo principal. El clustering espectral (notebooks 05 a 07) trata a los artistas como una red: cada uno unido a los más parecidos. Este anexo arma esa red de forma explícita y la describe con medidas de topología.

Preguntas:

1. ¿Qué forma tiene la red de trayectorias parecidas?
2. ¿Los perfiles son comunidades de esa red?
3. ¿Un algoritmo de detección de comunidades sin k fijo (Louvain) encuentra los mismos grupos?
4. ¿Qué artistas hacen de puente entre perfiles?
5. ¿Qué dicen los autovalores (eigengap) sobre la cantidad de grupos?

**Entrada:** matrices DTW de los notebooks 04 y 06 y perfiles finales del notebook 07.

In [ ]:
# PAQUETES

library(data.table)
library(igraph)
library(mclust)

## 1. Configuración y carga

In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

ARCHIVO_DISTANCIAS_NIVEL <- file.path("..", "chartmetric_pilotos", "distancias_clustering.rds")
ARCHIVO_DISTANCIAS_FORMA <- file.path("..", "chartmetric_pilotos", "distancias_forma_crecimiento.rds")
ARCHIVO_PERFILES <- file.path("..", "artistas_ST_perfiles_finales.csv")
# Notebook y figuras viven en LWR_grafos_exploracion/; los datos, en la raíz del repo
FIGURES_DIR <- "."

# Afinidad: la misma del spectral (escala local = distancia al 7.o vecino)
VECINO_ESCALA_LOCAL <- 7L

# Red de vecinos: cada artista se une a sus 10 vecinos DTW mas cercanos
# (union simetrica: basta que uno elija al otro)
VECINOS_RED <- 10L

# Semilla principal del proyecto (primera derivada de 999199) y redes al azar de referencia
SEMILLA_PRINCIPAL <- 742457L
N_REDES_AZAR <- 10L
N_AUTOVALORES <- 10L
N_PUENTES <- 10L
UMBRAL_FRONTERA <- 30   # % de vecinos de otro perfil

# Paleta de Come Together, la misma del notebook 07
# Intensidad: amarillo, lila y celeste (paleta de Come Together).
# Forma: naranja y azul, vecinos del amarillo y el celeste, y verde
# para el salto en YouTube, un color que no se usa en otro lado.
PALETA_RELLENO <- c(amarillo = "#FFF97A", lila = "#C6AEF6", celeste = "#47B3F7",
                    naranja = "#FFD66B", azul = "#6E8EF0", verde = "#7DD8A2")
PALETA_LINEA <- c(amarillo = "#C9B800", lila = "#8B6FD6", celeste = "#1E8FD8",
                  naranja = "#E0A100", azul = "#2F55C9", verde = "#1E9E5A")
COLOR_PERFIL <- list(
  intensidad = c(estable = "amarillo", moderado = "lila", fuerte = "celeste"),
  forma = c(parejo = "naranja", "salto en YouTube" = "verde", enfriamiento = "azul")
)
COLOR_TEXTO <- "#1D1E2C"

guardar_figura <- function(nombre, funcion_grafico, ancho, alto) {
  funcion_grafico()
  png(file.path(FIGURES_DIR, nombre), width = ancho, height = alto, res = 200)
  funcion_grafico()
  dev.off()
  invisible(nombre)
}

In [ ]:
# ============================================================
# CARGA: matrices DTW y perfiles finales
# ============================================================

MATRICES <- list(
  intensidad = readRDS(ARCHIVO_DISTANCIAS_NIVEL)$matrices$dtw,
  forma = readRDS(ARCHIVO_DISTANCIAS_FORMA)$matrices$dtw
)
ids_artistas <- rownames(MATRICES$intensidad)

perfiles <- fread(ARCHIVO_PERFILES)[, artist_id := as.character(artist_id)]
perfiles <- perfiles[match(ids_artistas, artist_id)]

PERFIL <- list(intensidad = perfiles$perfil_intensidad, forma = perfiles$perfil_forma)

# Sanity check: mismos artistas en el mismo orden y perfiles completos
stopifnot(
  identical(rownames(MATRICES$forma), ids_artistas),
  !anyNA(perfiles$artist_name),
  all(PERFIL$intensidad %in% names(COLOR_PERFIL$intensidad)),
  all(PERFIL$forma %in% names(COLOR_PERFIL$forma))
)
sapply(PERFIL, table)

## 2. Construcción de la red

Nodos: los 1252 artistas. Enlaces: cada artista con sus 10 vecinos DTW más cercanos. Peso: la afinidad del spectral, exp(-d² / σᵢσⱼ), de 0 a 1. La red completa de afinidades tiene un enlace entre cada par: la red de vecinos conserva sólo los fuertes, que son los que deciden los grupos.

In [ ]:
# ============================================================
# AFINIDAD Y RED DE VECINOS
# ============================================================

calcular_afinidad <- function(m) {
  escala <- apply(m, 1, function(fila) sort(fila)[VECINO_ESCALA_LOCAL + 1])
  afinidad <- exp(-m^2 / outer(escala, escala))
  diag(afinidad) <- 0
  afinidad
}

construir_red <- function(m, perfil) {
  afinidad <- calcular_afinidad(m)
  n <- nrow(m)
  vecinos <- t(apply(m, 1, function(fila) order(fila)[2:(VECINOS_RED + 1)]))
  enlaces <- unique(t(apply(cbind(rep(seq_len(n), each = VECINOS_RED), as.vector(t(vecinos))), 1, sort)))
  red <- graph_from_edgelist(enlaces, directed = FALSE)
  E(red)$weight <- afinidad[enlaces]
  E(red)$distancia <- m[enlaces]
  V(red)$name <- rownames(m)
  V(red)$artista <- perfiles$artist_name
  V(red)$perfil <- perfil
  red
}

AFINIDADES <- lapply(MATRICES, calcular_afinidad)
REDES <- Map(construir_red, MATRICES, PERFIL)

# Sanity check: todos los artistas, sin lazos ni enlaces repetidos, grado minimo = VECINOS_RED
stopifnot(
  all(sapply(REDES, vcount) == length(ids_artistas)),
  all(sapply(REDES, is_simple)),
  all(sapply(REDES, function(r) min(degree(r))) >= VECINOS_RED)
)
sapply(REDES, function(r) c(nodos = vcount(r), enlaces = ecount(r)))

## 3. Medidas globales

- **Densidad:** enlaces presentes sobre todos los posibles.
- **Transitividad:** probabilidad de que dos vecinos de un artista también sean vecinos entre sí (triángulos).
- **Camino medio y diámetro:** pasos típicos y máximos entre dos artistas.
- **Mundo pequeño:** transitividad y camino medio frente a redes al azar con los mismos nodos y enlaces. σ > 1 indica mundo pequeño: muy agrupada y aun así con caminos cortos.
- **Asortatividad por perfil:** de -1 a 1; positiva si los enlaces unen artistas del mismo perfil.
- **Modularidad:** de -0,5 a 1; cuánto más peso queda dentro de los grupos que lo esperado al azar. Desde 0,3 se considera estructura de comunidades marcada.

In [ ]:
# ============================================================
# MEDIDAS GLOBALES Y COMPARACION CON REDES AL AZAR
# ============================================================

medidas_globales <- rbindlist(lapply(names(REDES), function(representacion) {
  red <- REDES[[representacion]]
  grupos_perfil <- as.integer(factor(V(red)$perfil))

  set.seed(SEMILLA_PRINCIPAL)
  azar <- replicate(N_REDES_AZAR, {
    r <- sample_gnm(vcount(red), ecount(red))
    c(transitividad = transitivity(r, type = "global"), camino = mean_distance(r, weights = NA))
  })
  transitividad <- transitivity(red, type = "global")
  camino <- mean_distance(red, weights = NA)

  data.table(
    representacion = representacion,
    nodos = vcount(red),
    enlaces = ecount(red),
    densidad = round(edge_density(red), 4),
    grado_medio = round(mean(degree(red)), 1),
    componentes = components(red)$no,
    transitividad = round(transitividad, 3),
    transitividad_azar = round(mean(azar["transitividad", ]), 3),
    camino_medio = round(camino, 2),
    camino_medio_azar = round(mean(azar["camino", ]), 2),
    diametro = diameter(red, weights = NA),
    sigma_mundo_pequeno = round((transitividad / mean(azar["transitividad", ])) / (camino / mean(azar["camino", ])), 1),
    asortatividad_perfil = round(assortativity_nominal(red, grupos_perfil, directed = FALSE), 3),
    modularidad_perfiles = round(modularity(red, grupos_perfil, weights = E(red)$weight), 3)
  )
}))

medidas_globales

## 4. ¿Los perfiles son comunidades?

Dos miradas por perfil:

- **Matriz de mezcla:** a qué perfil van los enlaces de cada perfil (% de sus enlaces).
- **Conductancia:** peso que sale del perfil sobre el peso total del perfil. Cerca de 0, comunidad cerrada; cerca de 1, perfil disuelto en la red.

In [ ]:
# ============================================================
# MEZCLA ENTRE PERFILES Y CONDUCTANCIA
# ============================================================

extremos_enlaces <- function(red) {
  extremos <- ends(red, E(red), names = FALSE)
  data.table(desde = V(red)$perfil[extremos[, 1]], hasta = V(red)$perfil[extremos[, 2]], peso = E(red)$weight)
}

mezcla_perfiles <- rbindlist(lapply(names(REDES), function(representacion) {
  e <- extremos_enlaces(REDES[[representacion]])
  ambos_sentidos <- rbind(e, e[, .(desde = hasta, hasta = desde, peso)])
  ambos_sentidos[, .(enlaces = .N), by = .(desde, hasta)][
    , pct := round(100 * enlaces / sum(enlaces), 1), by = desde][, representacion := representacion]
}))

perfiles_comunidad <- rbindlist(lapply(names(REDES), function(representacion) {
  red <- REDES[[representacion]]
  fuerza <- strength(red)
  e <- extremos_enlaces(red)
  rbindlist(lapply(unique(V(red)$perfil), function(p) {
    peso_sale <- e[(desde == p) != (hasta == p), sum(peso)]
    en_perfil <- V(red)$perfil == p
    sub <- induced_subgraph(red, which(en_perfil))
    data.table(
      representacion = representacion, perfil = p, artistas = sum(en_perfil),
      grado_medio = round(mean(degree(red)[en_perfil]), 1),
      transitividad_interna = round(transitivity(sub, type = "global"), 3),
      componentes_internas = components(sub)$no,
      conductancia = round(peso_sale / sum(fuerza[en_perfil]), 3)
    )
  }))
}))

# Sanity check: en cada perfil los porcentajes de mezcla suman 100
stopifnot(all(abs(mezcla_perfiles[, sum(pct), by = .(representacion, desde)]$V1 - 100) < 0.5))

print(dcast(mezcla_perfiles, representacion + desde ~ hasta, value.var = "pct", fill = 0))
perfiles_comunidad[order(representacion, conductancia)]

## 5. Detección de comunidades sin k fijo: Louvain

Louvain agrupa nodos maximizando la modularidad y decide solo cuántas comunidades hay. Si encuentra grupos parecidos a los perfiles, la estructura no depende de haber fijado k = 3. El acuerdo se mide con el índice de Rand ajustado (ARI).

In [ ]:
# ============================================================
# LOUVAIN Y ACUERDO CON LOS PERFILES
# ============================================================

comunidades_louvain <- lapply(REDES, function(red) {
  set.seed(SEMILLA_PRINCIPAL)
  cluster_louvain(red, weights = E(red)$weight)
})

louvain_resumen <- rbindlist(lapply(names(REDES), function(representacion) {
  com <- comunidades_louvain[[representacion]]
  data.table(
    representacion = representacion,
    comunidades = length(com),
    tamanos = paste(sort(sizes(com), decreasing = TRUE), collapse = " / "),
    modularidad_louvain = round(modularity(com), 3),
    ari_con_perfiles = round(adjustedRandIndex(membership(com), PERFIL[[representacion]]), 3)
  )
}))

louvain_resumen

# Composicion: en que comunidades de Louvain cae cada perfil (% del perfil)
rbindlist(lapply(names(REDES), function(representacion) {
  data.table(representacion = representacion, perfil = PERFIL[[representacion]],
             comunidad = membership(comunidades_louvain[[representacion]]))
}))[, .N, by = .(representacion, perfil, comunidad)][
  , pct := round(100 * N / sum(N), 1), by = .(representacion, perfil)][
  pct >= 10][order(representacion, perfil, -pct)]

## 6. Puentes entre perfiles

**Intermediación:** cuántos caminos más cortos entre pares de artistas pasan por cada artista, con la distancia DTW como largo del enlace. Los de intermediación alta conectan zonas de la red. Un artista es de **frontera** si al menos el 30% de sus vecinos pertenece a otro perfil; los puentes son los de frontera con mayor intermediación.

In [ ]:
# ============================================================
# INTERMEDIACION: ARTISTAS PUENTE
# ============================================================

puentes <- rbindlist(lapply(names(REDES), function(representacion) {
  red <- REDES[[representacion]]
  intermediacion <- betweenness(red, weights = E(red)$distancia, normalized = TRUE)
  vecinos_otro_perfil <- sapply(seq_len(vcount(red)), function(i) {
    v <- neighbors(red, i)
    100 * mean(V(red)$perfil[v] != V(red)$perfil[i])
  })
  data.table(representacion = representacion, artista = V(red)$artista, perfil = V(red)$perfil,
             intermediacion = round(intermediacion, 4), pct_vecinos_otro_perfil = round(vecinos_otro_perfil))
}))

# Frontera: artistas con al menos 30% de vecinos de otro perfil
cat("Artistas de frontera por perfil:
")
print(puentes[, .(artistas = .N, frontera = sum(pct_vecinos_otro_perfil >= UMBRAL_FRONTERA),
                  pct_frontera = round(100 * mean(pct_vecinos_otro_perfil >= UMBRAL_FRONTERA), 1)), by = .(representacion, perfil)])

# Puentes: los de mayor intermediacion entre los de frontera
puentes[pct_vecinos_otro_perfil >= UMBRAL_FRONTERA][order(-intermediacion), head(.SD, N_PUENTES), by = representacion]

## 7. Autovalores y eigengap

Laplaciano normalizado de la red completa de afinidades (la misma que usa el spectral). Cada autovalor mide cuánto cuesta separar un grupo más; el **eigengap** es el salto más grande entre autovalores consecutivos y sugiere la cantidad de grupos. Se busca entre k = 2 y k = 8, el rango de las particiones.

In [ ]:
# ============================================================
# AUTOVALORES DEL LAPLACIANO NORMALIZADO
# ============================================================

autovalores <- rbindlist(lapply(names(AFINIDADES), function(representacion) {
  a <- AFINIDADES[[representacion]]
  grado <- rowSums(a)
  valores_afinidad <- eigen(a / sqrt(outer(grado, grado)), symmetric = TRUE, only.values = TRUE)$values
  laplaciano <- sort(1 - valores_afinidad)[seq_len(N_AUTOVALORES)]
  data.table(representacion = representacion, orden = seq_len(N_AUTOVALORES),
             autovalor = laplaciano, salto_al_siguiente = c(diff(laplaciano), NA))
}))

# Sanity check: el primer autovalor es cero (red conexa de afinidades)
stopifnot(all(abs(autovalores[orden == 1, autovalor]) < 1e-8))

eigengap <- autovalores[!is.na(salto_al_siguiente) & orden >= 2 & orden <= 8, .SD[which.max(salto_al_siguiente)], by = representacion]
print(autovalores[, .(representacion, orden, autovalor = round(autovalor, 4), salto_al_siguiente = round(salto_al_siguiente, 4))])
cat("\nEigengap (k sugerido = orden del autovalor antes del salto):\n")
eigengap[, .(representacion, k_sugerido = orden, salto = round(salto_al_siguiente, 4))]

## 8. Figuras

In [ ]:
# ============================================================
# FIGURA 1: la red de artistas, coloreada por perfil
# ============================================================

DISPOSICIONES <- lapply(REDES, function(red) {
  set.seed(SEMILLA_PRINCIPAL)
  layout_with_fr(red, weights = E(red)$weight, niter = 1500)
})

TITULOS_RED <- c(intensidad = "Intensidad: cuánto crece", forma = "Forma: cómo crece")

grafico_red <- function() {
  par(mfrow = c(1, 2), mar = c(4.5, 0.5, 3.5, 0.5), bg = "white", oma = c(0, 0, 3, 0))
  for (representacion in names(REDES)) {
    red <- REDES[[representacion]]
    tono <- COLOR_PERFIL[[representacion]][V(red)$perfil]
    plot(red, layout = DISPOSICIONES[[representacion]], vertex.label = NA,
         vertex.size = 2.2, vertex.color = PALETA_RELLENO[tono], vertex.frame.color = PALETA_LINEA[tono],
         edge.color = adjustcolor(COLOR_TEXTO, 0.07), edge.width = 0.5)
    title(TITULOS_RED[[representacion]], col.main = COLOR_TEXTO, cex.main = 1.3, font.main = 2)
    legend("bottom", inset = -0.08, horiz = TRUE, bty = "n", xpd = NA, pch = 21, pt.cex = 1.6,
           pt.bg = PALETA_RELLENO[COLOR_PERFIL[[representacion]]], col = PALETA_LINEA[COLOR_PERFIL[[representacion]]],
           legend = paste0(names(COLOR_PERFIL[[representacion]]), " (", table(V(red)$perfil)[names(COLOR_PERFIL[[representacion]])], ")"),
           text.col = COLOR_TEXTO)
  }
  mtext("La red de trayectorias parecidas: cada artista unido a sus 10 vecinos DTW más cercanos",
        outer = TRUE, cex = 1.25, font = 2, col = COLOR_TEXTO, line = 0.8)
}

guardar_figura("grafo_01_red_perfiles.png", grafico_red, ancho = 3200, alto = 1700)

In [ ]:
# ============================================================
# FIGURA 2: autovalores y eigengap
# ============================================================

grafico_eigengap <- function() {
  par(mfrow = c(1, 2), mar = c(4.5, 4.5, 3.5, 1), bg = "white", las = 1, oma = c(0, 0, 2.5, 0))
  for (rep_actual in names(AFINIDADES)) {
    a <- autovalores[representacion == rep_actual]
    salto <- eigengap[representacion == rep_actual]
    plot(a$orden, a$autovalor, type = "n", xlab = "Orden del autovalor", ylab = "Autovalor (laplaciano normalizado)",
         xaxt = "n", bty = "n", col.lab = COLOR_TEXTO, ylim = c(0, max(a$autovalor) * 1.08))
    axis(1, at = a$orden)
    abline(h = pretty(a$autovalor), col = "#EEEEEE")
    rect(salto$orden + 0.15, a$autovalor[salto$orden], salto$orden + 0.85, a$autovalor[salto$orden + 1],
         col = adjustcolor(PALETA_RELLENO[["celeste"]], 0.45), border = NA)
    lines(a$orden, a$autovalor, col = COLOR_TEXTO, lwd = 2)
    points(a$orden, a$autovalor, pch = 21, bg = "white", col = COLOR_TEXTO, cex = 1.4, lwd = 2)
    text(salto$orden + 0.95, mean(a$autovalor[salto$orden + 0:1]), paste0("eigengap\nk = ", salto$orden),
         pos = 4, col = PALETA_LINEA[["celeste"]], font = 2, xpd = NA)
    title(TITULOS_RED[[rep_actual]], col.main = COLOR_TEXTO, cex.main = 1.2)
  }
  mtext("Cuánto cuesta separar un grupo más: el salto más grande sugiere cuántos grupos hay",
        outer = TRUE, font = 2, cex = 1.15, col = COLOR_TEXTO, line = 0.5)
}

guardar_figura("grafo_02_eigengap.png", grafico_eigengap, ancho = 3000, alto = 1400)

## 9. Lectura

**Forma de la red.** Una sola componente: 1252 artistas, unos 10 mil enlaces, densidad de 1,3% a 1,4% y 16 a 17 vecinos por artista. Es un **mundo pequeño**: la transitividad (0,29 en intensidad, 0,25 en forma) es unas 20 veces la de una red al azar, y el camino medio (4,0 y 3,7 pasos) apenas supera el del azar (2,8). σ = 15,5 y 13,7.

**Los perfiles son comunidades.** Asortatividad por perfil de 0,75 y 0,71: tres de cada cuatro enlaces unen artistas del mismo perfil. En intensidad, el 80% al 90% de los enlaces de cada perfil queda adentro, y estables y fuertes casi no se tocan (0,7%): el moderado es el puente entre ambos. En forma, el parejo es casi cerrado (96,5% interno) y el enfriamiento el más abierto (26% de sus enlaces van al parejo). Conductancia de 0,02 a 0,18. La modularidad de los perfiles es marcada en intensidad (0,46) y baja en forma (0,17), porque el parejo reúne al 84% de los artistas.

**Louvain sin k fijo.** Encuentra 10 y 12 comunidades con modularidad mayor (0,65 y 0,62), y el acuerdo con los perfiles es bajo (ARI 0,21 y 0,04) porque parte más fino: los perfiles son uniones de comunidades. El 71% de los fuertes cae en una sola comunidad y el 92% del salto en YouTube en otra; estables, moderados y parejos se reparten en varias.

**Frontera y puentes.** Los de frontera (30% o más de vecinos de otro perfil) son el 13% de los estables, el 27% de los moderados y el 31% de los fuertes; en forma, el 45% del enfriamiento y el 4% de los parejos. Puentes con mayor intermediación: Cyndi Lauper y Deftones en intensidad; Tiago PZK en forma.

**Eigengap.** En intensidad marca k = 2: separar a los fuertes es el corte más barato, y el salto siguiente es igual de grande (0,11). En forma no hay un salto claro (todos entre 0,04 y 0,10; el mayor, en k = 7): el eigengap no decide, y por eso k se eligió con la separación entre perfiles y la lectura de las features.

**Síntesis.** Los perfiles no son cortes arbitrarios de un continuo: son regiones de una red de mundo pequeño, con fronteras porosas y un perfil intermedio que las une.